# CH3-2. DNN Training

## Preamble

In [7]:
# Standard Library
import time
import os
from pathlib import Path

# Data and Numerical Processing
import numpy as np
import pandas as pd

# SmartSim and SmartRedis
from smartsim import Experiment
from smartredis import Client

# Machine Learning and Modelling
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import onnx
import onnxruntime as ort
import jax2onnx
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_friedman1

# Visualisation
import DataGraph as dg
import matplotlib.pyplot as plt

In [8]:
# Remove inherited Slurm environment from the interactive parent job
for key in list(os.environ):
    if key.startswith(("SLURM_", "SBATCH_", "SRUN_")):
        os.environ.pop(key)

### Directories

In [9]:
# Directory
PROJECT_DIR = Path("/scratch/project_2015384/Hanseul")
BASE_DIR = PROJECT_DIR / "Codes" / "SmartSim"
MAIN_DIR = BASE_DIR / "OnlineInference"
DATASET_DIR = MAIN_DIR / "dataset"
SAVE_DIR = MAIN_DIR / "model"

for d in [PROJECT_DIR, BASE_DIR, MAIN_DIR, DATASET_DIR, SAVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Path
FILE_PATH = DATASET_DIR / "california_housing.csv"

### Configuration

In [10]:
# System configuration
SEED = 42
np.random.seed(SEED)
key = jax.random.PRNGKey(SEED)
N_JOBS = len(list(os.sched_getaffinity(0)))

# SmartRedis configuration
REDIS_PORT = 2026

In [11]:
# ONNX configuration (Backend ONNX ver: 1.21.0 -> opset: 26, IR: 13)
OPSET_VERSION = 26
IR_VERSION = 13

In [12]:
# Dataset configuration
N_SAMPLES = 10000
SIGMA_NOISE = [0.2, 0.5, 0.8]  # Noise levels for the Friedman dataset

### Metric

In [13]:
# Metric computation functions (MAE, MSE, R2)
def compute_metrics(model, X_test, y_test, verbose=True):
    # Compute metrics for the given model and test data
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Print the computed metrics
    summary = dg.TableMaker(
        title="Model Evaluation Metrics",
        columns=["Metric", "Value"],
    )
    summary.add_row("MAE", f"{mae:.4f}")
    summary.add_row("MSE", f"{mse:.4f}")
    summary.add_row("R2", f"{r2:.4f}")
    
    if verbose:
        summary.display()
    
    return (mae, mse, r2)

### Dataset I/O

In [14]:
# Generate the Friedman dataset with specified noise levels
def generate_friedman_dataset(n_samples, n_features, noise_levels, random_state=SEED):
    datasets = {}
    for noise in noise_levels:
        X, y = make_friedman1(n_samples=n_samples, n_features=n_features, noise=noise, random_state=random_state)
        datasets[noise] = (X, y)
    return datasets

datasets = generate_friedman_dataset(N_SAMPLES, n_features=10, noise_levels=SIGMA_NOISE)

# Combine all datasets into a single DataFrame for analysis
X = np.vstack([datasets[noise][0] for noise in SIGMA_NOISE])
y = np.hstack([datasets[noise][1] for noise in SIGMA_NOISE])

# Split the combined dataset into training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

#Scale the features using StandardScaler
#scaler = StandardScaler()
#X_train = scaler.fit_transform(X_train)
#X_test = scaler.transform(X_test)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)

## ML Regression

### ML Architecture

In [15]:
class Linear(eqx.Module):
    weight: jax.Array
    bias: jax.Array

    def __init__(self, key, in_features, out_features, init="glorot"):
        if isinstance(init, str):
            init = {
                "glorot": jax.nn.initializers.glorot_normal(),
                "he": jax.nn.initializers.he_normal(),
                "lecun": jax.nn.initializers.lecun_normal(),
            }.get(init.lower(), jax.nn.initializers.glorot_normal())

        self.weight = init(key, (out_features, in_features))
        self.bias = jnp.zeros(out_features)

    def __call__(self, x):
        return self.weight @ x + self.bias

In [16]:
class DNN(eqx.Module):
    layers: list
    activation: callable = eqx.field(static=True)

    def __init__(self, key, input_dim, hidden_dims, output_dim,
                 activation=jax.nn.silu, init="glorot"):

        self.layers = self._build_layers(
            layer=Linear,
            key=key,
            in_dim=input_dim,
            hidden_dims=hidden_dims,
            out_dim=output_dim,
            init=init
        )

        self.activation = activation

    @staticmethod
    def _build_layers(layer, key, in_dim, hidden_dims, out_dim, init="glorot"):
        dims = [in_dim] + hidden_dims + [out_dim]
        keys = jax.random.split(key, len(dims) - 1)

        return [
            layer(
                key=keys[i],
                in_features=dims[i],
                out_features=dims[i + 1],
                init=init
            )
            for i in range(len(dims) - 1)
        ]

    @staticmethod
    def _forward(layers, x, activation):
        for layer in layers[:-1]:
            x = activation(layer(x))

        return layers[-1](x)

    def __call__(self, x):
        return self._forward(self.layers, x, self.activation)

    def predict(self, X):
        return jax.vmap(self)(X).squeeze(-1)

In [17]:
def mse_loss(model, X, y):
    pred = model(X)
    return (pred - y) ** 2

def batch_loss(model, X, y):
    losses = jax.vmap(mse_loss, in_axes=(None, 0, 0))(model, X, y)
    return jnp.mean(losses)

@eqx.filter_value_and_grad
def compute_loss(model, X, y):
    return batch_loss(model, X, y)

@eqx.filter_jit
def train_step(model, opt_state, X, y, optimizer):
    loss, grads = compute_loss(model, X, y)
    updates, opt_state = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_array))
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

@eqx.filter_jit
def evaluate(model, X, y):
    preds = model.predict(X)
    mse = jnp.mean((y - preds) ** 2)
    mae = jnp.mean(jnp.abs(y - preds))
    return mse, mae

In [18]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_model = None
        self.should_stop = False

    def update(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_model = model
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self

    def get_best_model(self):
        return self.best_model

In [19]:
# Metric computation functions (MAE, MSE, R2)
def compute_metrics(model, X_test, y_test, verbose=True):
    # Compute metrics for the given model and test data
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Print the computed metrics
    summary = dg.TableMaker(
        title="Model Evaluation Metrics",
        columns=["Metric", "Value"],
    )
    summary.add_row("MAE", f"{mae:.4f}")
    summary.add_row("MSE", f"{mse:.4f}")
    summary.add_row("R2", f"{r2:.4f}")
    
    if verbose:
        summary.display()
    
    return (mae, mse, r2)

### Train and Inference

In [20]:
# Define a regression model configuration
input_dim = 10  # Number of features in the dataset
hidden_dims = [128, 128]
output_dim = 1

# Training parameters
activation = jax.nn.relu
lr_init = 1e-3
epochs = 2000
batch_size = 256
scheduler = optax.exponential_decay(init_value=lr_init, decay_rate=0.9, transition_steps=200)
optimizer = optax.adam(scheduler)

In [21]:
# Initialise model
model = DNN(key=key, input_dim=input_dim, hidden_dims=hidden_dims, output_dim=output_dim, activation=activation)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

In [22]:
# Training loop with early stopping
history = {'train_mse': [], 'train_mae': [], 'val_mse': [], 'val_mae': []}
early_stopping = EarlyStopping(patience=50, min_delta=1e-4)

training_table = dg.TableMaker(
    title="DNN Training Progress",
    columns=["Epoch", "Train MSE", "Train MAE", "Val MSE", "Val MAE", "Status"],
    mode="live"
)

for epoch in range(epochs):
    key, subkey = jax.random.split(key)
    perm = jax.random.permutation(subkey, len(X_train))
    X_shuffled = X_train[perm]
    y_shuffled = y_train[perm]

    for i in range(0, len(X_shuffled), batch_size):
        X_batch = X_shuffled[i:i+batch_size]
        y_batch = y_shuffled[i:i+batch_size]
        model, opt_state, loss = train_step(model, opt_state, X_batch, y_batch, optimizer)

    train_mse, train_mae = evaluate(model, X_train, y_train)
    val_mse, val_mae = evaluate(model, X_val, y_val)

    train_mse, train_mae = float(train_mse), float(train_mae)
    val_mse, val_mae = float(val_mse), float(val_mae)

    history['train_mse'].append(train_mse)
    history['train_mae'].append(train_mae)
    history['val_mse'].append(val_mse)
    history['val_mae'].append(val_mae)

    early_stopping.update(val_mse, model)
    status = "(best)" if early_stopping.counter == 0 else f"({early_stopping.counter}/{early_stopping.patience})"

    if epoch % 20 == 0 or early_stopping.should_stop:
        training_table.add_row(
            f"{epoch+1}/{epochs}",
            f"{train_mse:.4f}",
            f"{train_mae:.4f}",
            f"{val_mse:.4f}",
            f"{val_mae:.4f}",
            status
        )

    if early_stopping.should_stop:
        break

# Restore best model
model = early_stopping.get_best_model()

Epoch,Train MSE,Train MAE,Val MSE,Val MAE,Status
1/2000,15.3928,3.1428,15.2616,3.1495,(best)
21/2000,1.4062,0.8544,1.4333,0.8607,(best)
41/2000,0.5854,0.5650,0.5849,0.5606,(best)
61/2000,0.4675,0.5091,0.4713,0.5083,(best)
81/2000,0.4340,0.4924,0.4372,0.4926,(best)
101/2000,0.4177,0.4823,0.4215,0.4829,(1/50)
121/2000,0.4115,0.4786,0.4154,0.4794,(2/50)
141/2000,0.4088,0.4775,0.4122,0.4782,(best)
161/2000,0.4074,0.4766,0.4110,0.4773,(2/50)
181/2000,0.4069,0.4762,0.4104,0.4769,(3/50)


In [23]:
# Evaluate the DNN regressor model
metrics = compute_metrics(model, X_test, y_test)

Metric,Value
MAE,0.4819
MSE,0.4164
R2,0.9830


### Convert to ONNX and Save Model

In [24]:
def export_activation(X):
    if model.activation is jax.nn.relu:
        return jnp.maximum(X, 0)
    elif model.activation is jax.nn.tanh:
        return jnp.tanh(X)
    elif model.activation is jax.nn.silu:
        return X * jax.nn.sigmoid(X)
    else:
        raise ValueError(f"Unsupported activation: {model.activation}")

In [29]:
# Define batched inference function for ONNX model
def batch_forward(X):
    for layer in model.layers[:-1]:
        X = X @ layer.weight.T + layer.bias
        X = export_activation(X)

    return X @ model.layers[-1].weight.T + model.layers[-1].bias

# Export Equinox model to ONNX
onnx_model_dnn = jax2onnx.to_onnx(
    batch_forward,
    inputs=[("B", input_dim)],
    model_name="dnn_regressor",
    opset=OPSET_VERSION,
)

# Set ONNX IR version for RedisAI compatibility
onnx_model_dnn.ir_version = IR_VERSION

# Validate and save ONNX model
onnx.checker.check_model(onnx_model_dnn)
onnx.save(onnx_model_dnn, SAVE_DIR / "dnn_regressor.onnx")

INFO:root:Converting JAX function to ONNX model with parameters: model_name=dnn_regressor, opset=26, input_shapes=[('B', 10)], input_params=None
INFO:root:Dynamic batch dimensions detected.


In [26]:
# Check the equivalence of the original model and the exported ONNX model
X_check = jnp.asarray(X_test[:32], dtype=jnp.float32)

y_model = jax.vmap(model)(X_check)
y_export = batch_forward(X_check)

print(np.allclose(y_model, y_export, rtol=1e-6, atol=1e-7))
print(np.max(np.abs(np.asarray(y_model - y_export))))

True
0.0


### Save Model as eqx

In [27]:
# Save the trained model using eqx
eqx.tree_serialise_leaves(SAVE_DIR / "dnn_regressor.eqx", model)